In [ ]:
!pip install pyspark py4j

In [42]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import to_date, mean, col, coalesce

spark = SparkSession.builder.appName("Read CSV Example").getOrCreate()

# Чтение CSV-файла
book_df = spark.read.csv("/content/drive/MyDrive/Colab Notebooks/books.csv", header=True, inferSchema=True)
author_df = spark.read.csv("/content/drive/MyDrive/Colab Notebooks/authors.csv", header=True, inferSchema=True)

book_df = book_df.select(book_df.book_id, book_df.title, book_df.author_id, book_df.genre, book_df.price, to_date(book_df.publish_date, "yyyy-MM-dd").alias("publish_date"))
author_df = author_df.select(author_df.author_id, author_df.name, to_date(author_df.birth_date, "yyyy-MM-dd").alias("birth_date"), author_df.country)

df = book_df.join(author_df, on="author_id", how="inner")

df.createOrReplaceTempView("book_data")

top_author_df = spark.sql("""
SELECT author_id, name, sum(price) as total_revenue
FROM book_data
group by author_id, name
order by 3 desc
limit 5
""")

count_books_df = spark.sql("""
SELECT genre, count(book_id) as cnt
FROM book_data
group by genre
order by 2 desc
""")

avg_price_df = spark.sql("""
SELECT author_id, name, avg(price) as average_price
FROM book_data
group by author_id, name
""")

book_2000_df = spark.sql("""
SELECT *
FROM book_data
where year(publish_date) > 2000
order by price desc
""")

# Показ результатов
top_author_df.show() # Найдите топ-5 авторов, книги которых принесли наибольшую выручку.
count_books_df.show() # Найдите количество книг в каждом жанре.
avg_price_df.show() # Подсчитайте среднюю цену книг по каждому автору.
book_2000_df.show() # Найдите книги, опубликованные после 2000 года, и отсортируйте их по цене.



+---------+--------+-------------+
|author_id|    name|total_revenue|
+---------+--------+-------------+
|        2|Author_2|       231.97|
|        7|Author_7|       132.66|
|        1|Author_1|       111.86|
|        8|Author_8|       107.16|
|        5|Author_5|        88.83|
+---------+--------+-------------+

+-----------+---+
|      genre|cnt|
+-----------+---+
|Non-Fiction|  9|
|    Science|  3|
|    Fiction|  3|
|    Fantasy|  3|
|    Mystery|  2|
+-----------+---+

+---------+---------+-----------------+
|author_id|     name|    average_price|
+---------+---------+-----------------+
|        4| Author_4|             83.7|
|        5| Author_5|            88.83|
|        8| Author_8|            35.72|
|        6| Author_6|           43.965|
|        2| Author_2|          57.9925|
|        7| Author_7|            44.22|
|       10|Author_10|           21.165|
|        1| Author_1|37.28666666666667|
|        9| Author_9|            46.31|
+---------+---------+-----------------+

